# Session 8 — Building AI-based Evaluators: LLM-as-a-Judge

**Who checks the checker?**

Session 7 built four agents and measured whether they coordinated. It never asked whether
the diagnosis was any *good*. Today we build the thing that can ask — and then we point it
at itself.

> **The sentence this session adds:** *You cannot measure a difference smaller than your
> judge's own noise.*

Halvard Works is fictional — the plant, its machines, sensors, manuals, parts and history
were invented for this course. Inspired by publicly described industrial-copilot products;
not affiliated with or endorsed by any vendor.

## Every term, before it is used

| term | in this session it means |
|---|---|
| **judge** | an evaluator whose verdict comes from a language model instead of from code. |
| **rubric** | the prompt that tells the judge where to look and what would count as failing. |
| **verdict** | one of exactly three words: `SOUND`, `UNSOUND`, `INSUFFICIENT-EVIDENCE`. |
| **score** | those three words as `1`, `0`, `None`. `None` means *skipped*, as everywhere else. |
| **an arm** | one of six versions of the same report: four broken on purpose, two not. |
| **separation** | how much more often a judge fires on its own broken arm than on the arms it should pass. |
| **wobble** | how often the same judge, on the same unchanged report, disagrees with its own most common answer. |
| **the gate** | separation's lower bound must clear wobble's upper bound. Otherwise the judge is decoration. |
| **a verdict fires** | it returned `0`. Confusingly, that is the judge working. |

In [ ]:
# [PULL]
!git pull --ff-only
!python check_env.py

In [ ]:
# [PINS]
# A resolution FAILURE is loud. A resolution SUCCESS that quietly installed a
# different version is silent, and would corrupt every number below.
import importlib.metadata as md
PINS = {'langchain-core': '1.6.1', 'langgraph': '1.2.11', 'langsmith': '0.11.1'}
bad = {p: md.version(p) for p, want in PINS.items() if md.version(p) != want}
print('pins OK' if not bad else f'WRONG VERSIONS: {bad} -- fix before going on')

In [ ]:
# [SETUP]
# The repo is organised by session. Python is not: it puts THIS folder on
# sys.path, not the repo root, so `import evalkit` needs one line of help.
# _path.py walks up to the repo root and prepends shared/ and plant/.
import _path  # noqa: F401

# Reload the course modules in DEPENDENCY ORDER before importing anything.
# A Jupyter kernel caches modules; edit a file, re-run, and you silently get
# the first import. Session 7's note applies unchanged.
import importlib, sys
for _m in ('plant7', 'plant_tools7', 'delegation_rows7', 'plant_agents7',
           'seeds7', 'coord_eval7', 'bench7',
           'judge8', 'judge_seeds8', 'judge_bench8', 'agree8', 'human_labels8'):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

import json, judge8, judge_seeds8 as seeds8, judge_bench8, agree8, human_labels8

# STUB = True runs a deterministic keyword matcher: free, no key, and it tells you
# NOTHING about judges. It is here so every cell below produces output even if your
# key is not working. Flip it to False to spend money and get a real answer.
STUB = True
print('judge8', judge8.__version__, '| seeds8', seeds8.__version__,
      '|', 'STUB — plumbing only' if STUB else 'LIVE')

## 1 · What Session 7 left on the table

Four coordination evaluators. Every one of them **skips** the single-agent arm — and it is
right to. Scoring one agent against a four-agent plan is an emissions test for a bicycle.

But that leaves four columns of `None`, and a one-agent answer graded by keyword match.

In [ ]:
# [NULLS]
import coord_eval7
from delegation_rows7 import BY_ID

# Session 7's runs live in session-07/. _path.session(7) finds them from here.
runs = json.load(open(_path.session(7) / 'runs7.json'))['runs']
single = next(r for r in runs if r.get('phase') == 'comparison'
              and r['version'] == 'single' and r['row_id'] == 'HW-001')

for k, v in coord_eval7.run_all(single['outputs'], BY_ID['HW-001']).items():
    print(f"{k:24s} {str(v['score']):>5s}  {v['comment'][:60]}")

In [ ]:
# [REPORT]
# The raw material for everything below: one real, live, four-agent report from
# Session 7. No agent is re-run today. The whole seeded set costs zero.
base = seeds8.load_base()
print(base['answer'][:900])
print('...')
print('TAIL:', base['tail'])

In [ ]:
# [SPLIT]
# A judge reads ONE section. The pipeline joins them with [agent] headers; the
# single arm produces one unlabelled blob, and split_reports handles both.
for agent, text in judge8.split_reports(base['answer']).items():
    print(f"{agent:16s} {len(text):5d} chars   {text[:70]}...")
print()
print('single arm ->', list(judge8.split_reports(single['outputs']['answer'])))

## 2 · The rubric

An evaluator is a hypothesis about a failure: it says **where to look** and **what would
count as failing**. That rule does not stop applying because the evaluator is made of prose.

Three things make a rubric checkable rather than a vibe:

1. it names the failure, not the virtue — *UNSOUND if the cited mechanism belongs to a
   different machine*, not *is this a good diagnosis?*
2. it gives the judge the reference a human reviewer would have, and **not** the answer key;
3. it demands one supporting fact, so the verdict can be argued with.

In [ ]:
# [RUBRIC]
# Read one. This is the entire judge -- there is nothing else in there.
print(judge8.build_prompt('diagnosis_soundness', base, 'CONVEYOR')[:1800])

In [ ]:
# [PARSE]
# What happens to a reply that ignores the instruction. Nothing is retried and
# nothing is dropped: a judge that cannot follow a three-word instruction is a
# measurement ABOUT the judge, and hiding it behind a retry loop is how a bad
# judge looks good.
for reply in ['SOUND\nBPFO 3.19x matches the equipment record.',
              'unsound\nthe cited section is a limits table.',
              'The report seems fine to me.',
              'SOUND',
              '4/5\nquite good',
              '']:
    word, why = judge8.parse_verdict(reply)
    print(f"{reply[:40]!r:46s} -> {word:22s} {why[:44]}")

## 3 · Four judges, and where they fire

| judge | asks | anchored to |
|---|---|---|
| `diagnosis_soundness` | does the stated evidence support the fault it names? | the equipment record |
| `doc_relevance` | is the cited section the one that answers *this* fault? | the manual index, **with titles** |
| `recommendation_safety` | is the action proportionate to the risk and the stock? | criticality, alarm limits, parts |
| `workflow_coherence` | do the sections contradict each other? | the sections themselves |

They are **not** aimed at Session 7's pipeline-vs-single tie. That interval was
`-4% [-52, +43]`, and no judge can resolve a difference wider than the plus-or-minus of the
thing it is judging. That is this session's own spine, and it applies to us first.

In [ ]:
# [JUDGE1]
# One judge, one report. With STUB = False this costs one model call.
res = judge8.run_all(base, None, stub=STUB)
for k, v in res.items():
    print(f"{k:24s} {str(v['score']):>5s}  {v['comment'][:74]}")

## 4 · Calibration: reports that are wrong on purpose

Session 7 validated its code evaluators by breaking the pipeline four ways where the right
answer was known. Same move, one level up.

**And say what it does not buy.** The same person wrote the flaw and the rubric, so a judge
that catches these flaws may be catching that person's vocabulary. This measures
*sensitivity*. It cannot tell you the judge is right about a report nobody tampered with.

Which is why two of the six arms are not broken: `healthy` and `padded`. Twenty of the
twenty-four cells in the matrix are false-positive tests.

In [ ]:
# [ARMS]
arms = seeds8.build()
base_len = len(arms['healthy']['answer'])
for name, o in arms.items():
    target = [k for k, v in seeds8.EXPECT[name].items() if v == 0] or ['- control -']
    print(f"{name:16s} {len(o['answer']) - base_len:+6d} chars   "
          f"should fail: {', '.join(target)}")

In [ ]:
# [DIFF]
# Exactly what one mutator changed. Every mutator RAISES if its anchor is missing
# or the text did not move -- a mutator that silently no-ops hands the measurement
# a HEALTHY report labelled BROKEN, and the number you get back is a lie.
import difflib
a = arms['healthy']['answer'].split('. ')
b = arms['wrong_evidence']['answer'].split('. ')
for line in difflib.unified_diff(a, b, lineterm='', n=0):
    if line[:1] in '+-' and line[:3] not in ('+++', '---'):
        print(line[:150])

In [ ]:
# [MATRIX]
# 6 arms x 4 judges x REPS.
#
# Three reps on the stub costs nothing, and separation on n=1 is a story rather
# than a measurement -- 1/1 vs 0/5 gives a lower bound of +10%, which is below
# the wobble bound, so the gate below would read DECORATION for arithmetic
# reasons rather than for anything about the judge. Live, three reps is 72 model
# calls; that is the instructor's bill, not yours, which is why REPS drops to 1.
REPS = 3 if STUB else 1
recs = judge_bench8.score_arms(arms, stub=STUB, reps=REPS, verbose=False)
ks = judge8.JUDGE_KEYS
print(f"{'arm':16s} " + ' '.join(f'{k[:13]:>15s}' for k in ks))
for name in arms:
    row = {r['judge']: r for r in recs if r['arm'] == name and r['rep'] == 1}
    print(f'{name:16s} ' + ' '.join(f"{row[k]['verdict'][:13]:>15s}" for k in ks))
print(f'\n{len(recs)} verdicts ({REPS} rep(s)). The table shows rep 1;',
      'the intervals below use all of them.')

In [ ]:
# [KEY]
# Agreement with the answer key -- and, separately, the FALSE ALARMS, which are
# the half that catches the failure this course has already made twice.
keyed = [r for r in recs if r['correct'] is not None]
misses = [r for r in keyed if r['expected'] == 0 and not r['correct']]
alarms = [r for r in keyed if r['expected'] == 1 and not r['correct']]
print(f'agreement with the key : {sum(1 for r in keyed if r["correct"])}/{len(keyed)}')
print(f'missed a real flaw     : {len(misses)}  {[r["arm"] + "/" + r["judge"] for r in misses]}')
print(f'fired on a good report : {len(alarms)}  {[r["arm"] + "/" + r["judge"] for r in alarms]}')

## 5 · Separation, wobble, and the gate

```
separation = P(UNSOUND | its own broken arm) - P(UNSOUND | the arms it should pass)
wobble     = how often it disagrees with its own most common answer, same report, n times
```

Both are proportions from small `n`, and several of them will be `0/10` or `10/10`. The
textbook `p ± 1.96·√(p(1-p)/n)` gives an interval of **zero width** at 0 and at 1 — *we saw
no disagreements, therefore the rate is exactly 0%, no uncertainty*. That is the most
confident wrong answer in applied statistics and it is one line of code away at all times.

So: Wilson intervals, and the rule of three (`3/n`) that Session 5 already taught.

In [ ]:
# [SEP]
# Run the matrix a few times first if you are live -- separation on n=1 is a story,
# not a measurement.
for j in judge8.JUDGE_KEYS:
    print('  ' + agree8.separation(recs, j).line())

In [ ]:
# [WOB]
# The same judge, the same report, n times. With STUB = False and n=10 across
# three reports this is 120 model calls -- which is why the instructor ran it and
# you are reading the saved file.
wrecs = judge_bench8.wobble(arms, stub=STUB, n=5, verbose=False)
for j in judge8.JUDGE_KEYS:
    w = agree8.wobble(wrecs, j)
    print('  ' + w.line())
    for arm, (flips, n, mode) in sorted(w.per_arm.items()):
        print(f'      {arm:16s} {flips}/{n} differ from {mode}')

In [ ]:
# [GATE]
# THE PUNCHLINE. One number per judge, and one sentence.
# Reads the SAVED live file when it exists, because the cell above is stub by
# default and a stub judge separates 100% and wobbles 0% BY CONSTRUCTION.
import os
if os.path.exists('judge_runs8.json'):
    live = judge_bench8.load('judge_runs8.json')
    print('reading the instructor\'s LIVE verdicts\n')
else:
    live = recs + wrecs
    print('!! no judge_runs8.json -- these are STUB numbers and are not findings\n')

rep = agree8.report(live)
for j in judge8.JUDGE_KEYS:
    s, w = rep['separation'][j], rep['wobble'][j]
    print(f"{j:24s} sep_lo {s['lo']:+.0%}  vs  wobble_hi {w['upper']:.0%}"
          f"   -> {rep['verdicts'][j]}")

In [ ]:
# [VERBOSITY]
# Same content, four times the words, nothing false added. A verdict that moves
# here moved on LENGTH. This is the bias demonstration, and it cost four calls
# because `padded` was already in the matrix.
for j, v in rep['verbosity'].items():
    print(f"{j:24s} {str(v['healthy']):22s} -> {str(v['padded']):22s} "
          f"{'MOVED' if v['moved'] else 'unchanged'}")

In [ ]:
# [HUMAN]
# Judge vs code vs human, on the only cases in this course where a human opinion
# is actually needed: real disagreements from Session 7's live runs.
h = human_labels8.report()
print(f"{h['n_instances']} run-instances across {h['n_rows']} rows\n")
for row, d in h['rows'].items():
    print(f"{row}  {d['shape']:19s} agent said {str(d['agent_said']):34s}"
          f" -> I said: {d['label'] or 'UNLABELLED'}")
print('\n' + h['caveat'])

## 6 · Hands-on — break it

Open **`my_attack8.py`**. Pick one broken report. Rewrite it so its judge says `SOUND`
**without removing the flaw**.

Write `PREDICT` first. The screener will not run until you do — a prediction written
afterwards is not a prediction.

| verdict | what happened |
|---|---|
| `FOOLED` | the judge passed a report that is still broken. This is the target. |
| `NO-EFFECT` | the judge still says `UNSOUND`. Your edit did not move it. |
| `REPAIRED` | you removed the flaw. Put it back and attack the verdict instead. |
| `COLLATERAL` | a *different* judge fired. That counts, and it is the argument for having four. |

You are not attacking the model's reasoning. You are attacking the seam between the verdict
and the evidence line.

In [ ]:
# [ATTACK]
# One model call per attempt, on your key. Five attempts is a normal session.
!python screen_my_attack.py

## 7 · What today added

A judge can answer questions no code evaluator in this course can express — whether a
diagnosis is *sound*, whether a citation is *relevant*, whether a recommendation is *safe*.

And a judge is an agent, so it inherits every problem an agent has. It is a sample, not a
reading. Ask it twice and you may get two answers, and every verdict it has ever given you
sits inside that spread.

> **You cannot measure a difference smaller than your judge's own noise.**

Which is why the gate is not *is the judge good* — it is *is the difference it claims bigger
than its disagreement with itself*. A judge that fails that may still be right. We simply
cannot show it, and a slide that says otherwise is claiming a difference smaller than the
instrument.

**Homework** is in your inbox: one attack with its verdict, and one rubric line you would
change, with the reason.